![logog](https://raw.githubusercontent.com/Pacific-AI-Corp/langtest/main/docs/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pacific-AI-Corp/langtest/blob/main/demo/tutorials/llm_notebooks/dataset-notebooks/MTS_Dialog.ipynb)

**LangTest** is an open-source python library designed to help developers deliver safe and effective Natural Language Processing (NLP) models. Whether you are using **John Snow Labs, Hugging Face, Spacy** models or **OpenAI, Cohere, AI21, Hugging Face Inference API and Azure-OpenAI** based LLMs, it has got you covered. You can test any Named Entity Recognition (NER), Text Classification, fill-mask, Translation model using the library. We also support testing LLMS for Question-Answering, Summarization and text-generation tasks on benchmark datasets. The library supports 100+ out of the box tests. For a complete list of supported test categories, please refer to the [documentation](http://langtest.org/docs/pages/docs/test_categories).


# Getting started with LangTest

In [ ]:
%pip install "langtest[openai]==2.7.0"

# Harness and Its Parameters

The Harness class is a testing class for Natural Language Processing (NLP) and LLM models. It evaluates the performance of a NLP model on a given task using test data and generates a report with test results.Harness can be imported from the LangTest library in the following way.

In [ ]:
#Import Harness from the LangTest library
from langtest import Harness

### Set environment for OpenAI

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"

## MTS Dialog Dataset

In [ ]:
from langtest.types import HarnessConfig


config: HarnessConfig = {
    "evaluation": {
        "model": "gpt-4o-mini",
        "hub": "openai",
        "metric": "llm_eval",
        "threshold": 9,

    },
    "tests": {
        "defaults": {
            "min_pass_rate": 0.8,
        },
        "clinical": {
            "clinical_note_summary": {
                "dataset_path": "mts-dialog",
                "num_samples": 50,    
            }
        }
    }
}

In [5]:
harness = Harness(
    task="summarization",
    model=
        {
        "model": "gpt-4.1-mini",
        "hub": "openai",
        "type": "chat"
    },
    data=[],
    config=config
)

Test Configuration : 
 {
 "evaluation": {
  "model": "gpt-4o-mini",
  "hub": "openai",
  "metric": "llm_eval",
  "threshold": 9
 },
 "tests": {
  "defaults": {
   "min_pass_rate": 0.8
  },
  "clinical": {
   "clinical_note_summary": {
    "dataset_path": "mts-dialog",
    "num_samples": 50
   }
  }
 }
}


### Generating the test cases.

In [ ]:
harness.generate()

In [7]:
testcases = harness.testcases()
testcases.head()

,category,test_type,dialogue
0,clinical,clinical_note_summary,Doctor: I'm glad to hear that your Afib is und...
1,clinical,clinical_note_summary,"Doctor: Do you have a history of tobacco, alco..."
2,clinical,clinical_note_summary,"Doctor: Hey, bud. What brings you in today? \r..."
3,clinical,clinical_note_summary,"Doctor: Hello, this is my assistant, and she w..."
4,clinical,clinical_note_summary,Doctor: Do you have any prior history of surge...


harness.generate() method automatically generates the test cases (based on the provided configuration)

### Running the tests

In [ ]:
harness.run()

Called after harness.generate() and is to used to run all the tests.  Returns a pass/fail flag for each test.

### Generated Results

In [9]:
results = harness.generated_results()
results.head()

,category,test_type,dialogue,expected_result,actual_result,feedback,pass
0,clinical,clinical_note_summary,Doctor: I'm glad to hear that your Afib is und...,ASSESSMENT Section: \nChronic atrial fibrillat...,ASSESSMENT: \nThe patient has atrial fibrilla...,"{'Factual Completeness': 10, 'No Hallucination...",True
1,clinical,clinical_note_summary,"Doctor: Do you have a history of tobacco, alco...",FAM/SOCHX Section: \nDenies history of Tobacco...,FAM/SOCHX: The patient reports no history of t...,"{'Factual Completeness': 10, 'No Hallucination...",True
2,clinical,clinical_note_summary,"Doctor: Hey, bud. What brings you in today? \r...",ROS Section: \nAs mentioned denies any orophar...,ROS Section:\nThe patient presents with a rash...,"{'Factual Completeness': 10, 'No Hallucination...",True
3,clinical,clinical_note_summary,"Doctor: Hello, this is my assistant, and she w...",GENHX Section: \nMs. ABC returns today for fol...,GENHX: Miss A B C is a patient with a history ...,"{'Factual Completeness': 10, 'No Hallucination...",True
4,clinical,clinical_note_summary,Doctor: Do you have any prior history of surge...,"PASTSURGICAL Section: \nBack surgery, shoulder...",PASTSURGICAL: The patient has a history of bac...,"{'Factual Completeness': 10, 'No Hallucination...",True


This method returns the generated results in the form of a pandas dataframe, which provides a convenient and easy-to-use format for working with the test results. You can use this method to quickly identify the test cases that failed and to determine where fixes are needed.

### Final Results

We can call `.report()` which summarizes the results giving information about pass and fail counts and overall test pass/fail flag.

In [11]:
harness.report()

,category,test_type,fail_count,pass_count,pass_rate,minimum_pass_rate,pass
0,clinical,clinical_note_summary,2,48,96%,80%,True
